# Text Classification on Social Media Commentary

Assignment devoloped by António Romão (up202108704), Guilherme Gonçalves (up202107768) and Pedro Leitão (up202107852) for the Natural Language Processing course at FEUP.

## Table of Contents
1. [Introduction](#1-introduction)
    - 1.1. [Context and Problem Statement](#11-context-and-problem-statement)
    - 1.2. [Assignment Objectives](#12-assignment-objectives)
2. [Data Provenance and Characteristics](#2-data-provenance-and-characteristics)
    - 2.1. [Origin and Collections Methods](#21-origin-and-collection-methods)
    - 2.2. [Annotation Methodology](#22-annotation-methodology)
    - 2.3. [Label Definitions](#23-label-definitions)
    - 2.4. [Initial Processing and Structure](#24-initial-processing-and-structure)
3. [Exploratory Data Analysis (EDA)](#3-exploratory-data-analysis-eda)
4. [Data Pre-processing](#4-data-pre-processing)
5. [Feature Extraction and Representation](#5-feature-extraction-and-representation)
6. [Model Training and Selection](#6-model-training-and-selection)
7. [Model Evaluation](#7-model-evaluation)
8. [Error Analysis](#8-error-analysis)
9. [Related Work Comparison](#9-related-work-comparison)
10. [Conclusion](#10-conclusion)

## 1. Introduction

[[go back to top]](#table-of-contents)

This report presents the development and evaluation of Natural Language Processing (NLP) classifiers designed for a specialized text classification task. The introduction is divided into two main parts: first, it establishes the context and outlines the core problem by exploring the nuances of the provided social media commentary dataset; second, it details the specific academic objectives, scope, and technical constraints required for this assignment.

### 1.1. Context and Problem Statement

[[go back to section]](#1-introduction)

The landscape of online discourse is complex, and traditional sentiment analysis, often limited to simple 'positive', 'negative', or 'neutral' labels, is frequently insufficient for capturing the true nature of social media commentary. To better understand how users interact, debate, and communicate online, it is necessary to analyze the rhetorical purpose and communicative intent behind the text. 

The dataset provided for this assignment addresses this need by categorizing social media comments into five distinct, nuanced labels: Argumentative, Informational, Opinion, Expressive, and Neutral. These comments were specifically gathered using the search queries 'politics' or 'US Politics'. To ensure a diverse representation of online behavior, the data was sourced from multiple platforms with varying community norms, specifically YouTube, Hacker News, MetaFilter, Reddit, and BlueSky. The collection spans a timeframe from 2024 up to mid-February 2026.

**Problem Statement**: The core problem to be addressed in this assignment is the development of an effective Natural Language Processing (NLP) pipeline for text classification. The objective is to build robust machine learning classifiers capable of automatically categorizing these political social media comments into one of the five target labels. Successfully solving this problem involves navigating the inherent noise and variability of social media text through careful application of data pre-processing, feature extraction (including both sparse and dense representations), and traditional machine learning algorithms.

### 1.2. Assignment Objectives

[[go back to section]](#1-introduction)

The primary aim of this assignment is to build effective Natural Language Processing (NLP) classifiers for a specific text classification task. To achieve this, the project involves a comprehensive exploration of the standard NLP pipeline, including data pre-processing, feature extraction, and a comparative analysis of sparse versus dense feature representations, such as word embeddings. Prior to model development, a critical objective is to conduct an exploratory data analysis (EDA) to thoroughly understand the dataset, which includes documenting class sizes and word distributions (e.g., TF-IDF) supported by proper visualizations.

A core constraint of this project is the strict focus on "traditional" machine learning algorithms. The models explored will include classifiers such as Naive Bayes, Logistic Regression, Decision Trees, Random Forest, Support Vector Machines (SVM), Multi-Layer Perceptrons (MLP), or XGBoost. The implementation of any deep learning architectures based on Convolutional Neural Networks (CNNs), Recurrent Neural Networks (RNNs), or Transformers is explicitly prohibited for this assignment.

Finally, a rigorous evaluation methodology is required. This involves establishing a simple baseline model to benchmark future developments and reporting classifier performance using appropriate metrics such as Precision, Recall, F1, and macro-F1 scores. We will also undertake a detailed error analysis on the best-performing models to understand the reasons behind misclassifications. Furthermore, the project aims to frame our work by comparing our classifier results with any existing related work on the same problem, placing our achievements in the context of prior published research.

## 2. Data Provenance and Characteristics

[[go back to top]](#table-of-contents)

Before developing our text classifiers, the assignment requires us to establish a clear understanding of our data's provenance and inherent characteristics. To achieve this, we will examine the ADS509 Dataset, which serves as the foundation for this project. This dataset was specifically designed to capture the nuanced nature of social media commentary, moving beyond traditional 'positive', 'negative', or 'neutral' sentiment labels. The following subsections detail the sources, annotation methodology, label definitions, and initial processing steps used to construct this dataset, providing the necessary context for our subsequent exploratory data analysis and feature engineering.

### 2.1. Origin and Collection Methods

[[go back to section]](#2-data-provenance-and-characteristics)

To effectively classify the communicative intent behind online discourse, the data for this assignment was sourced from five distinct social media platforms: YouTube, Hacker News, MetaFilter, Reddit, and BlueSky. By utilizing multiple platforms, the dataset captures a wider variety of community norms and conversational styles. 

The collection focused specifically on political discussions, utilizing the targeted search queries 'politics' or 'US Politics'. The overall timeframe for the collected data ranges from 2024 to mid-February of 2026. However, the exact chronological span varies depending on the specific platform's traffic volume and API rate limits. For example, due to its heavy traffic, the Reddit collection hit daily rate limits and is consequently restricted to posts from just the first two weeks of February 2026. In contrast, data pulled from MetaFilter extends as far back as 2024.

Furthermore, specific structural filtering criteria were applied during the collection process to ensure data quality and balance:
* Posts containing fewer than 10 comments were entirely excluded from the dataset.
* A maximum threshold of 300 comments was extracted from any single post to prevent highly active or viral threads from disproportionately dominating the data distribution.

### 2.2. Annotation Methodology

[[go back to section]](#2-data-provenance-and-characteristics)


The annotation process for this dataset employed a hybrid approach, combining initial human validation with large-scale automated labeling using Large Language Models (LLMs). 

First, a manual pilot annotation phase was conducted to establish a baseline. A sample of 100 comments was independently labeled. These initial human annotations were subsequently compared and revised to resolve disagreements and solidify the labeling criteria.

For the remainder of the dataset, the labeling was scaled up using a Batch API to query three distinct state-of-the-art language models: Gemini Flash 3, Chat GPT 5.1, and Claude Haiku 4.5. 

To ensure the models accurately captured the nuanced definitions of the five classes, a few-shot prompting strategy was utilized. Specifically, the prompt provided to the LLMs included:
* 10 examples of correctly labeled samples to demonstrate the expected output.
* 10 examples of previously incorrectly labeled samples, alongside their corrected labels, to help the models avoid common classification pitfalls.

Finally, a consensus mechanism was applied for quality control. Only comments that achieved agreement from two or more of the three LLMs were retained in the final dataset. The remaining comments that failed to reach this majority consensus were set aside for further evaluation and excluded from this core dataset.

### 2.3. Label Definitions

[[go back to section]](#2-data-provenance-and-characteristics)

The dataset categorizes social media comments into five distinct, nuanced labels, moving beyond simple sentiment analysis. Understanding the boundaries between these classes is crucial for our text classification task:

* **Argumentative:** These comments make specific claims, predictions, or assertions that are supported by reasoning. They often use evidence, anecdotes, or scenarios to build a case. The key distinction from an 'Opinion' is the clear attempt to persuade or explain the reasoning behind a position, rather than just stating it.
* **Informational:** This category includes comments that share facts, data, links, or relevant context. They typically exhibit low emotional affect, aiming solely to inform rather than to convince or react. This includes answering questions with factual content. Unlike 'Argumentative' comments, 'Informational' ones present data without advocating for a specific stance.
* **Opinion:** These comments state a value judgment, stance, or "take" without providing substantial reasoning. Examples include simple assertions like "This is good/bad/wrong/overrated". They differ from 'Argumentative' text by lacking a real attempt to support the claim, and from 'Expressive' text because they do make a specific point rather than just reacting.
* **Expressive:** This label captures emotional reactions, sarcasm, jokes, venting, and exclamations. The primary intent is to express feeling rather than to make a substantive point. It includes performative agreements or disagreements (e.g., "THIS," "lol exactly," "what a joke"). The key distinction from 'Opinion' is the absence of an identifiable stance, consisting purely of affect.
* **Neutral:** This category acts as a catch-all for comments that do not clearly fit into the other four categories. It encompasses clarifying or rhetorical questions, meta-commentary, off-topic remarks, and simple factual questions directed at other users.

### 2.4. Initial Processing and Structure

[[go back to section]](#2-data-provenance-and-characteristics)

Before applying our own specific NLP pre-processing techniques, it is important to acknowledge the initial cleaning steps and structural formatting already performed by the dataset creators. 

**Data Processing Steps:**
To ensure a cleaner baseline for text classification, the following processing actions were executed before the dataset was finalized:
* Approximately 2,000 to 3,000 duplicate comments were removed.
* Any rows containing NaN (Not a Number) values were dropped.
* Emojis were systematically converted into their corresponding text representations using the [emoji package](https://pypi.org/project/emoji/).
* All text was converted to lowercase to maintain uniformity.
* Remaining HTML artifacts were stripped from the comments.
* URL links were replaced with a generic `[URL]` tag to normalize the text and prevent the models from overfitting to specific web addresses.
* Escaped characters (e.g., `&/quot;`) were converted back to their standard text equivalents (e.g., `"`).

**Dataset Structure:**
The resulting dataset consists of two primary features: `text` (the social media comment as a string) and `label` (the assigned numeric class ID). The specific mapping of labels to their corresponding numeric IDs is as follows:
* **0**: Neutral
* **1**: Opinion
* **2**: Argumentative
* **3**: Expressive
* **4**: Informational

It is pre-divided into three standard splits to facilitate model training and evaluation:
* **Train Split:** 49,268 examples.
* **Valid Split:** 10,557 examples.
* **Test Split:** 10,558 examples.

## 3. Exploratory Data Analysis (EDA)

[[go back to top]](#table-of-contents)

## 4. Data Pre-processing

[[go back to top]](#table-of-contents)

## 5. Feature Extraction and Representation

[[go back to top]](#table-of-contents)

## 6. Model Training and Selection

[[go back to top]](#table-of-contents)

## 7. Model Evaluation

[[go back to top]](#table-of-contents)

## 8. Error Analysis

[[go back to top]](#table-of-contents)

## 9. Related Work Comparison

[[go back to top]](#table-of-contents)

## 10. Conclusion

[[go back to top]](#table-of-contents)